In [15]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/jianghongab/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


# Daily Feature Pipeline for Grass Pollen (Stockholm)

## Sections:
1. Fetch Pollen Data from Pollenrapporten API
2. Insert into Feature Group

**Schedule this notebook to run daily during pollen season (May-August)**

In [16]:
import datetime
import pandas as pd
import hopsworks
from mlfs.airquality import util
import warnings
import json
warnings.filterwarnings("ignore")

## Connect to Hopsworks

In [17]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()
secrets = hopsworks.get_secrets_api()

location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
country=location['country']
city=location['city']
street=location['street']

# Stockholm coordinates
latitude = location['latitude']
longitude = location['longitude']

today = datetime.date.today()
print(f"Fetching pollen data for: {today}")

2026-01-11 22:24:45,030 INFO: Closing external client and cleaning up certificates.
2026-01-11 22:24:45,032 INFO: Connection closed.
2026-01-11 22:24:45,033 INFO: Initializing external client
2026-01-11 22:24:45,034 INFO: Base URL: https://c.app.hopsworks.ai:443


2026-01-11 22:24:46,335 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436
Fetching pollen data for: 2026-01-11


## Get Feature Group Reference

In [18]:
grass_pollen_fg = fs.get_feature_group(name="grass_pollen", version=1)
weather_fg = fs.get_feature_group(name="weather", version=1)

## Fetch Today's Pollen Data

In [19]:
# Fetch last 7 days to ensure we get today's data
start_date = (today - datetime.timedelta(days=7)).strftime('%Y-%m-%d')
end_date = today.strftime('%Y-%m-%d')

pollen_df = util.get_historical_pollen(
    start_date=start_date,
    end_date=end_date
)

# Convert to datetime_id format
if not pollen_df.empty:
    pollen_df['datetime_id'] = pd.to_datetime(pollen_df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')
    pollen_df = pollen_df.drop(columns=['date'])
    # Get today's data
    today_str = today.strftime('%Y-%m-%d')
    pollen_today = pollen_df[pollen_df['datetime_id'].str.startswith(today_str)]
else:
    # Create default pollen data with 0 value for today
    pollen_today = pd.DataFrame({'datetime_id': [today.strftime('%Y-%m-%d %H:%M:%S')], 'grass_pollen': [0]})

if len(pollen_today) > 0:
    print(f"Pollen data retrieved: {len(pollen_today)} records")
    print(pollen_today)
else:
    print("No pollen data available (likely outside monitoring season)")

No pollen data available for 2026-01-04 to 2026-01-11
Pollen data retrieved: 1 records
           datetime_id  grass_pollen
0  2026-01-11 00:00:00             0


Get Weather Forecast data

In [20]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city

daily_df['day_of_year'] = daily_df['date'].dt.dayofyear
daily_df['month'] = daily_df['date'].dt.month
daily_df['is_high_season'] = daily_df['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# Add GDD (Growing Degree Days) feature: Max(0, (T_mean - T_base))
T_base = 5.0  # Base temperature for grass growth
daily_df['gdd_daily'] = daily_df['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
daily_df['gdd_cumsum'] = daily_df.groupby(daily_df['date'].dt.year)['gdd_daily'].cumsum()


# Add lagged weather features (yesterday's weather affects today's pollen)
daily_df['precip_lag_1'] = daily_df['precipitation_sum'].shift(1)
daily_df['temp_lag_1'] = daily_df['temperature_2m_mean'].shift(1)
daily_df['wind_lag_1'] = daily_df['wind_speed_10m_max'].shift(1)
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')
daily_df = daily_df.rename(columns={'date': 'datetime_id'}) # 改名
daily_df

Coordinates 59.25°N 18.0°E
Elevation 24.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,datetime_id,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1
0,2026-01-04 00:00:00,-5.526500,0.0,19.813087,24.702408,Stockholm,4,1,0,0,0,NaN,NaN,NaN
1,2026-01-05 00:00:00,-7.326500,0.0,10.630672,28.300659,Stockholm,5,1,0,0,0,0.0,-5.526500,19.813087
2,2026-01-06 00:00:00,-1.926500,0.1,3.415260,71.564964,Stockholm,6,1,0,0,0,0.0,-7.326500,10.630672
3,2026-01-07 00:00:00,-7.976501,0.0,6.915374,231.340164,Stockholm,7,1,0,0,0,0.1,-1.926500,3.415260
4,2026-01-08 00:00:00,0.173500,0.0,21.674870,168.503464,Stockholm,8,1,0,0,0,0.0,-7.976501,6.915374
5,2026-01-09 00:00:00,-2.976500,0.0,14.113653,84.144089,Stockholm,9,1,0,0,0,0.0,0.173500,21.674870
6,2026-01-10 00:00:00,-5.526500,0.0,21.962950,359.060822,Stockholm,10,1,0,0,0,0.0,-2.976500,14.113653
7,2026-01-11 00:00:00,-3.426500,0.0,24.014996,347.005371,Stockholm,11,1,0,0,0,0.0,-5.526500,21.962950
8,2026-01-12 00:00:00,-3.926500,0.0,6.162207,353.290253,Stockholm,12,1,0,0,0,0.0,-3.426500,24.014996
9,2026-01-13 00:00:00,-1.876500,0.0,9.585739,124.286934,Stockholm,13,1,0,0,0,0.0,-3.926500,6.162207


## Upload to Feature Store

In [21]:
if not pollen_today.empty:
    grass_pollen_fg.insert(pollen_today, wait=True)
    print("✅ Pollen data uploaded successfully")
else:
    print("⚠️ No data to upload")

Uploading Dataframe: 100.00% |█| Rows 1/1 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: grass_pollen_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/grass_pollen_1_offline_fg_materialization/executions
2026-01-11 22:25:07,250 INFO: Waiting for execution to finish. Current state: INITIALIZING. Final status: UNDEFINED
2026-01-11 22:25:10,444 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-01-11 22:27:49,377 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-01-11 22:27:49,538 INFO: Waiting for log aggregation to finish.
2026-01-11 22:28:14,966 INFO: Execution finished successfully.


Online data ingestion progress: 0.00% |          | Rows 0/471

✅ Pollen data uploaded successfully


In [22]:
# Insert new data
daily_df = daily_df.drop(columns=['day_of_year', 'month', 'is_high_season', 'gdd_daily', 'gdd_cumsum', 'precip_lag_1', 'temp_lag_1','wind_lag_1'], errors='ignore')
weather_fg.insert(daily_df, wait=True)

2026-01-11 22:28:15,469 INFO: 	2 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1292436/fs/1265790/fg/1911320


Uploading Dataframe: 100.00% |█| Rows 14/14 | Elapsed Time: 00:01 | Remaining Time: 00:


Launching job: weather_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/weather_1_offline_fg_materialization/executions
2026-01-11 22:28:33,590 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-01-11 22:28:39,961 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-01-11 22:30:37,725 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-01-11 22:30:44,102 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-01-11 22:30:44,259 INFO: Waiting for log aggregation to finish.
2026-01-11 22:31:19,700 INFO: Execution finished successfully.


Online data ingestion progress: 0.00% |          | Rows 0/701

(Job('weather_1_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "precipitation_sum",
           "min_value": -0.1,
           "max_value": 1000.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 808034
         }
       },
       "result": {
         "observed_value": 0.0,
         "element_count": 14,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2026-01-11T09:28:15.000468Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     },
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column